# FPL API Ingestion
This notebook focuses on fetching data from the Fantasy Premier League (FPL) API and storing it in Parquet format for efficient processing in subsequent notebooks.

## 1. Install Required Libraries
We need the `requests` library to interact with the FPL API, `pandas` for data manipulation, and `pyarrow` for Parquet file handling.

In [1]:
import requests
import pandas as pd
import os
from datetime import datetime

## 2. API Interaction
The `fetch_fpl_data` function sends GET requests to the FPL API endpoints and returns the JSON response.

In [2]:
def fetch_fpl_data(endpoint):
    """Fetches data from the FPL API.

    Args:
        endpoint (str): The API endpoint to fetch.

    Returns:
        dict: The JSON response from the API.
    """

    base_url = "https://fantasy.premierleague.com/api/"
    url = base_url + endpoint
    response = requests.get(url)
    response.raise_for_status()  # Raise an exception for bad status codes (4xx or 5xx)
    return response.json()

## 3. Fetch Data from Various Endpoints
We fetch data from the `bootstrap-static`, `element-summary`, and `fixtures` endpoints.

In [3]:
# Fetch data from various endpoints
bootstrap_data = fetch_fpl_data("bootstrap-static/")
elements_data = bootstrap_data['elements']
teams_data = bootstrap_data['teams']
events_data = bootstrap_data['events']

# Convert to Pandas DataFrames
elements_df = pd.DataFrame(elements_data)
teams_df = pd.DataFrame(teams_data)
events_df = pd.DataFrame(events_data)

# Fetch player summary data
player_summary_data = {}
for player_id in elements_df['id']:
    player_summary_data[player_id] = fetch_fpl_data(f"element-summary/{player_id}/")

# Extract history from player summary data
player_history_data = {}
for player_id, data in player_summary_data.items():
    player_history_data[player_id] = data['history']

# Convert player history to a DataFrame
player_history_dfs = []
for player_id, history in player_history_data.items():
    temp_df = pd.DataFrame(history)
    temp_df['element'] = player_id  # Add player ID as a column
    player_history_dfs.append(temp_df)

player_history_df = pd.concat(player_history_dfs, ignore_index=True)

# Fetch fixture data
fixture_data = {}
for team_id in teams_df['id']:
    fixture_data[team_id] = fetch_fpl_data(f"fixtures/?team={team_id}")

# Extract fixture data from team fixture data
fixtures_dfs = []
for team_id, fixtures in fixture_data.items():
    temp_df = pd.DataFrame(fixtures)
    temp_df['team_id'] = team_id
    fixtures_dfs.append(temp_df)

fixtures_df = pd.concat(fixtures_dfs, ignore_index=True)

## 4. Data Structuring and Storage
We create a directory to store the data and save each DataFrame as a Parquet file.

In [4]:
# Create a directory for storing data (if it doesn't exist)
data_source = "fpl_api"
current_datetime = datetime.now().strftime("%Y%m%d_%H%M%S")
data_dir = f"../data/raw/{data_source}/{current_datetime}"
if not os.path.exists(data_dir):
    os.makedirs(data_dir)

# Save DataFrames as CSV files
elements_df.to_csv(os.path.join(data_dir, "2024_25_elements.csv"), index=False)
teams_df.to_csv(os.path.join(data_dir, "2024_25_teams.csv"), index=False)
events_df.to_csv(os.path.join(data_dir, "2024_25_events.csv"), index=False)
player_history_df.to_csv(os.path.join(data_dir, "2024_25_player_history.csv"), index=False)
fixtures_df.to_csv(os.path.join(data_dir, "2024_25_fixtures.csv"), index=False)

print(f"Data ingestion complete. Data saved to {data_dir}.")

Data ingestion complete. Data saved to ../data/raw/fpl_api/20250406_161919.
